### This .ipynb file is a combination of all scripts from task4_part1_2a - src folder. It is created to make it convenient to reproduce the results.

### There are also little code adjustments in some lines, including:
- Add HAC clustering on TF-IDF

- Find Silhouette score for HAC

- Adjust "interpret_clusters" function to not only save the results from embedding but also TF-IDF, as it is neccessary for comparison and evaluation

- Adjust the input for "analyze_impact" function. This doesn't impact the result at all. An extra input is needed because of the different saved file named I changed in "interpret_clusters" function

These adjustments are made to get the data and results needed for comparison and evaluation.

## Data Exploration

In [1]:
import pandas as pd
from datasets import load_dataset
import os

# Create a results directory if it doesn't exist
os.makedirs('results', exist_ok=True)

print("Loading dataset...")
dataset = load_dataset("Tobi-Bueck/customer-support-tickets")
df = pd.DataFrame(dataset['train'])

print(f"Total records: {len(df)}")

# Record raw missingness before filling text fields
missing_values = df.isnull().sum()

# Create text column
df['subject'] = df['subject'].fillna('')
df['body'] = df['body'].fillna('')
df['text'] = df['subject'] + "\n" + df['body']

# Split by language
df_en = df[df['language'] == 'en'].copy()
df_de = df[df['language'] == 'de'].copy()

print(f"English records: {len(df_en)}")
print(f"German records: {len(df_de)}")

# Sampling 5000 per language
if len(df_en) > 5000:
    df_en = df_en.sample(5000, random_state=42)
if len(df_de) > 5000:
    df_de = df_de.sample(5000, random_state=42)

print(f"Sampled English: {len(df_en)}")
print(f"Sampled German: {len(df_de)}")

# EDA Analysis
df['text_length'] = df['text'].astype(str).str.len()
duplicate_values = df.duplicated(subset=['subject', 'body']).sum()
priority_dist = df['priority'].value_counts()
type_dist = df['type'].value_counts()
text_len_desc = df['text_length'].describe()

# Save basic stats
with open('results/data_stats.txt', 'w', encoding='utf-8') as f:
    f.write("Task 4: Exploring Data Distributions\n")
    f.write("====================================\n\n")
    f.write(f"Total records: {len(df)}\n")
    f.write(f"Duplicate records (based on subject+body): {duplicate_values}\n\n")
    f.write(f"Missing Values per Column:\n{missing_values.to_string()}\n\n")
    f.write(f"Priority Distribution:\n{priority_dist.to_string()}\n\n")
    f.write(f"Type Distribution:\n{type_dist.to_string()}\n\n")
    f.write(f"Text Length (characters) Distribution:\n{text_len_desc.to_string()}\n\n")
    f.write(f"English sampled: {len(df_en)}\n")
    f.write(f"German sampled: {len(df_de)}\n")
    f.write("\nTop Queues (EN):\n" + df_en['queue'].value_counts().head(5).to_string() + "\n")
    f.write("\nTop Queues (DE):\n" + df_de['queue'].value_counts().head(5).to_string() + "\n")

# Save sampled data
df_en.to_csv('results/df_en_sampled.csv', index=False)
df_de.to_csv('results/df_de_sampled.csv', index=False)

print("\nData exploration and sampling complete.")

Loading dataset...
Total records: 61765
English records: 28261
German records: 33504
Sampled English: 5000
Sampled German: 5000

Data exploration and sampling complete.


## Preprocessing

In [ ]:
# %pip install spacy

In [ ]:
# !python -m spacy download en_core_web_sm

In [ ]:
# !python -m spacy download de_core_news_sm

In [2]:
import pandas as pd
import spacy
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
import pickle

# Create a results/processed directory if it doesn't exist
os.makedirs('results/processed', exist_ok=True)

# Load sampled data
df_en = pd.read_csv('results/df_en_sampled.csv')
df_de = pd.read_csv('results/df_de_sampled.csv')

# Preprocessing function
def preprocess(texts, nlp_model):
    processed_texts = []
    # Using pipe for efficiency
    for doc in nlp_model.pipe(texts, disable=['ner', 'parser']):
        # Lowercase, remove punct/stopwords/digits, lemmatize
        tokens = [token.lemma_.lower() for token in doc if not token.is_stop and not token.is_punct and not token.is_digit and len(token.text) > 2]
        processed_texts.append(" ".join(tokens))
    return processed_texts

print("Starting preprocessing...")

# Load Spacy models
print("Loading Spacy EN...")
nlp_en = spacy.load('en_core_web_sm')
print("Loading Spacy DE...")
nlp_de = spacy.load('de_core_news_sm')

# Process English
print("Processing English texts...")
df_en['text'] = df_en['text'].fillna('').astype(str)
df_en['processed_text'] = preprocess(df_en['text'], nlp_en)

# Process German
print("Processing German texts...")
df_de['text'] = df_de['text'].fillna('').astype(str)
df_de['processed_text'] = preprocess(df_de['text'], nlp_de)

# Vectorization - TF-IDF
print("Vectorizing TF-IDF...")
tfidf_en = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
X_tfidf_en = tfidf_en.fit_transform(df_en['processed_text'])

tfidf_de = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
X_tfidf_de = tfidf_de.fit_transform(df_de['processed_text'])

# Vectorization - Embeddings
print("Vectorizing Embeddings...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
X_emb_en = model.encode(df_en['text'].tolist(), show_progress_bar=True)
X_emb_de = model.encode(df_de['text'].tolist(), show_progress_bar=True)

# Save processed data and vectors
print("Saving processed data and vectors...")
df_en.to_csv('results/processed/df_en_preprocessed.csv', index=False)
df_de.to_csv('results/processed/df_de_preprocessed.csv', index=False)

with open('results/processed/vectors_en.pkl', 'wb') as f:
    pickle.dump({'tfidf': X_tfidf_en, 'emb': X_emb_en, 'tfidf_model': tfidf_en}, f)

with open('results/processed/vectors_de.pkl', 'wb') as f:
    pickle.dump({'tfidf': X_tfidf_de, 'emb': X_emb_de, 'tfidf_model': tfidf_de}, f)

print("Preprocessing and vectorization complete.")


Starting preprocessing...
Loading Spacy EN...
Loading Spacy DE...
Processing English texts...
Processing German texts...
Vectorizing TF-IDF...
Vectorizing Embeddings...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Saving processed data and vectors...
Preprocessing and vectorization complete.


## Clustering Analysis

In [3]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
print
# Create results/clustering directory
os.makedirs('results/clustering', exist_ok=True)

def run_clustering_analysis(lang, vectors, vector_name):
    print(f"Running clustering for {lang} - {vector_name}...")
    
    # Range of k
    k_range = range(5, 26, 2)
    sse = []
    silhouette_avg = []
    
    # K-Means analysis
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(vectors)
        sse.append(kmeans.inertia_)
        
        # Silhouette score (sampled if too large for speed)
        if vectors.shape[0] > 10000:
            sample_idx = np.random.choice(vectors.shape[0], 10000, replace=False)
            score = silhouette_score(vectors[sample_idx], kmeans.labels_[sample_idx])
        else:
            score = silhouette_score(vectors, kmeans.labels_)
        silhouette_avg.append(score)
        print(f"k={k}, SSE={kmeans.inertia_:.2f}, Silhouette={score:.4f}")
    
    # Save metrics
    metrics = pd.DataFrame({
        'k': k_range,
        'sse': sse,
        'silhouette': silhouette_avg
    })
    metrics.to_csv(f'results/clustering/metrics_{lang}_{vector_name}.csv', index=False)
    
    return metrics

# Load vectors
print("Loading vectors...")
with open('results/processed/vectors_en.pkl', 'rb') as f:
    v_en = pickle.load(f)
with open('results/processed/vectors_de.pkl', 'rb') as f:
    v_de = pickle.load(f)

# Run k-mean for English
run_clustering_analysis('en', v_en['tfidf'], 'tfidf')
run_clustering_analysis('en', v_en['emb'], 'emb')

# Run k-mean for German
run_clustering_analysis('de', v_de['tfidf'], 'tfidf')
run_clustering_analysis('de', v_de['emb'], 'emb')


print("Running HAC...")

# Run HAC for English - TF-IDF
hac_en = AgglomerativeClustering(n_clusters=15, linkage='ward')
hac_labels_en_tfidf = hac_en.fit_predict((v_en['tfidf']).toarray())
print("HAC for EN - TF-IDF: ", hac_labels_en_tfidf)  
score_en_tfidf = silhouette_score(v_en['tfidf'].toarray(), hac_labels_en_tfidf)
print("Silhouette Score: ", score_en_tfidf)
# Run HAC for English - Embeding
hac_labels_en_emb = hac_en.fit_predict(v_en['emb'])
print("HAC for EN - Emb: ", hac_labels_en_emb)  
score_en_emb = silhouette_score(v_en['emb'], hac_labels_en_emb)
print("Silhouette Score: ", score_en_emb)

# Run HAC on German - TF-IDF
hac_de = AgglomerativeClustering(n_clusters=15, linkage='ward')
hac_labels_de_tfidf = hac_de.fit_predict((v_de['tfidf']).toarray())
print("HAC for DE - TF-IDF: ", hac_labels_de_tfidf) 
score_de_tfidf = silhouette_score(v_de['tfidf'].toarray(), hac_labels_de_tfidf)
print("Silhouette Score: ", score_de_tfidf) 
# Run HAC on German - Embedding
hac_labels_de_emb = hac_de.fit_predict(v_de['emb'])
print("HAC for DE - Emb: ", hac_labels_de_emb)  
score_de_emb = silhouette_score(v_de['emb'], hac_labels_de_emb)
print("Silhouette Score: ", score_de_emb) 

# Save HAC labels
np.save('results/clustering/hac_labels_en_tfidf.npy', hac_labels_en_tfidf)
np.save('results/clustering/hac_labels_de_tfidf.npy', hac_labels_de_tfidf)
np.save('results/clustering/hac_labels_en_emb.npy', hac_labels_en_emb)
np.save('results/clustering/hac_labels_de_emb.npy', hac_labels_de_emb)

print("Clustering analysis complete.")


Loading vectors...
Running clustering for en - tfidf...
k=5, SSE=4530.82, Silhouette=0.0275
k=7, SSE=4465.59, Silhouette=0.0248
k=9, SSE=4420.17, Silhouette=0.0261
k=11, SSE=4384.61, Silhouette=0.0265
k=13, SSE=4355.35, Silhouette=0.0249
k=15, SSE=4337.45, Silhouette=0.0268
k=17, SSE=4308.61, Silhouette=0.0245
k=19, SSE=4284.22, Silhouette=0.0248
k=21, SSE=4259.68, Silhouette=0.0220
k=23, SSE=4244.42, Silhouette=0.0224
k=25, SSE=4225.78, Silhouette=0.0241
Running clustering for en - emb...
k=5, SSE=32743.46, Silhouette=0.1290
k=7, SSE=30982.15, Silhouette=0.1021
k=9, SSE=29448.20, Silhouette=0.1040
k=11, SSE=28573.46, Silhouette=0.1052
k=13, SSE=27895.77, Silhouette=0.1054
k=15, SSE=27410.89, Silhouette=0.1051
k=17, SSE=26946.49, Silhouette=0.0886
k=19, SSE=26583.46, Silhouette=0.0891
k=21, SSE=26158.74, Silhouette=0.0809
k=23, SSE=25919.36, Silhouette=0.0799
k=25, SSE=25660.34, Silhouette=0.0750
Running clustering for de - tfidf...
k=5, SSE=4600.53, Silhouette=0.0233
k=7, SSE=4543.81,

## Plot Stability of K values

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Create results/plots directory
os.makedirs('results/plots', exist_ok=True)

def plot_metrics(lang, vector_name):
    file_path = f'results/clustering/metrics_{lang}_{vector_name}.csv'
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return
    
    df = pd.read_csv(file_path)
    
    fig, ax1 = plt.subplots(figsize=(10, 6))

    # SSE Plot
    color = 'tab:blue'
    ax1.set_xlabel('Number of clusters (k)')
    ax1.set_ylabel('Inertia (SSE)', color=color)
    ax1.plot(df['k'], df['sse'], marker='o', color=color, label='SSE')
    ax1.tick_params(axis='y', labelcolor=color)

    # Silhouette Plot
    ax2 = ax1.twinx()
    color = 'tab:red'
    ax2.set_ylabel('Silhouette Score', color=color)
    ax2.plot(df['k'], df['silhouette'], marker='s', color=color, label='Silhouette')
    ax2.tick_params(axis='y', labelcolor=color)

    plt.title(f'Clustering Stability Analysis ({lang.upper()} - {vector_name.upper()})')
    fig.tight_layout()
    plt.savefig(f'results/plots/stability_{lang}_{vector_name}.png')
    plt.close()

# Plot for all combinations
print("Generating stability plots...")
plot_metrics('en', 'tfidf')
plot_metrics('en', 'emb')
plot_metrics('de', 'tfidf')
plot_metrics('de', 'emb')

# Generate Stability Conclusion
with open('results/clustering/stability_conclusion.md', 'w', encoding='utf-8') as f:
    f.write("# Clustering Stability & k-Selection Analysis\n\n")
    f.write("Plots generated successfully. The visual analysis of SSE (elbow method) and Silhouette scores will guide the final parameter selections (vectorization choice and k-value), which are discussed in detail in the main pipeline report.\n")

print("Plots and conclusion generated in results/plots/ and results/clustering/.")


Generating stability plots...
Plots and conclusion generated in results/plots/ and results/clustering/.


## BERTopic

In [ ]:
# CODE HERE

## LDA

In [ ]:
# CODE HERE

----------------------------------------------------------------

# NO LONGER IN USE

## Interpret Clusters

In [5]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer

# Create results/interpretation directory
os.makedirs('results/interpretation', exist_ok=True)

def get_ctfidf(texts, labels, n_clusters, lang='en'):
    """Calculate c-TF-IDF keywords for each cluster."""
    # Group texts by cluster
    documents = pd.DataFrame({'text': [str(t) if pd.notnull(t) else "" for t in texts], 'cluster': labels})
    documents_per_cluster = documents.groupby(['cluster'], as_index=False).agg({'text': ' '.join})
    
    # Calculate count of words per cluster
    if lang == 'en':
        stop_words_list = 'english'
    else:
        try:
            from spacy.lang.de.stop_words import STOP_WORDS
            stop_words_list = list(STOP_WORDS)
        except ImportError:
            stop_words_list = None
            
    count_vectorizer = CountVectorizer(stop_words=stop_words_list, min_df=2)
    count = count_vectorizer.fit_transform(documents_per_cluster.text)
    words = count_vectorizer.get_feature_names_out()
    
    # Calculate c-TF-IDF
    t = count.toarray()
    w = t.sum(axis=1)
    tf = t / w.reshape(-1, 1)
    
    idf = np.log(1 + (w.mean() / (t.sum(axis=0) + 1e-6)))
    ctfidf = tf * idf
    
    # Get top words
    top_words = {}
    for i in range(n_clusters):
        top_indices = ctfidf[i].argsort()[-10:][::-1]
        top_words[i] = [words[idx] for idx in top_indices]
    
    return top_words

def interpret_clusters(lang, df, vectors, k=15, method = 'emb'):
    print(f"Interpreting clusters for {lang}...")
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(vectors)
    df['cluster'] = labels
    
    # Get c-TF-IDF words
    top_words = get_ctfidf(df['processed_text'].tolist(), labels, k, lang)
    
    # Get representative tickets (closest to centroid)
    dist = kmeans.transform(vectors) # Distance to each centroid
    
    report = []

    def safe_mode(series):
        mode = series.dropna().mode()
        return mode.iloc[0] if not mode.empty else "N/A"

    for i in range(k):
        # Top 3 representative indices within the cluster
        cluster_indices = np.where(labels == i)[0]
        if len(cluster_indices) == 0:
            rep_tickets = []
        else:
            cluster_dist = dist[cluster_indices, i]
            closest_idx_within_cluster = cluster_indices[cluster_dist.argsort()[:3]]
            rep_tickets = df.iloc[closest_idx_within_cluster][['subject', 'queue', 'tag_1']].values.tolist()
        
        # Most frequent queue/tag in cluster
        cluster_df = df[df['cluster'] == i]
        top_queue = safe_mode(cluster_df['queue'])
        top_tag = safe_mode(cluster_df['tag_1'])
        
        cluster_info = {
            'cluster_id': i,
            'size': int((labels == i).sum()),
            'top_words': ", ".join(top_words[i]),
            'top_queue': top_queue,
            'top_tag': top_tag,
            'rep_tickets': rep_tickets
        }
        report.append(cluster_info)
    
    # Save report
    rep_df = pd.DataFrame(report)
    rep_df.to_csv(f'results/interpretation/cluster_report_{lang}_{method}.csv', index=False)
    
    # Save df with labels
    df.to_csv(f'results/interpretation/df_{lang}_{method}_labeled.csv', index=False)
    
    return rep_df

# Load data and vectors
print("Loading data and vectors...")
df_en = pd.read_csv('results/processed/df_en_preprocessed.csv')
df_de = pd.read_csv('results/processed/df_de_preprocessed.csv')

with open('results/processed/vectors_en.pkl', 'rb') as f:
    v_en = pickle.load(f)
with open('results/processed/vectors_de.pkl', 'rb') as f:
    v_de = pickle.load(f)

# Interpret
interpret_clusters('en', df_en, v_en['emb'], k=15, method = 'emb')
interpret_clusters('de', df_de, v_de['emb'], k=15, method = 'emb')
interpret_clusters('en', df_en, v_en['tfidf'], k=15, method = 'tfidf')
interpret_clusters('de', df_de, v_de['tfidf'], k=15, method = 'tfidf')

print("Interpretation complete.")


Loading data and vectors...
Interpreting clusters for en...
Interpreting clusters for de...
Interpreting clusters for en...
Interpreting clusters for de...
Interpretation complete.


## Labelling

In [6]:
import pandas as pd

# Create results/interpretation directory
os.makedirs('results/analysis', exist_ok=True)

en_labels = {
    0: "Project Management Tool Integration",
    1: "Digital Marketing Engagement Decline",
    2: "General/Miscellaneous (Noisy)",
    3: "Network Connectivity & Device Sync",
    4: "Medical Data Security Breaches",
    5: "SaaS Platform Integration Docs",
    6: "Hospital Data Protection",
    7: "Digital Strategy & Brand Growth",
    8: "Marketing Campaign Performance",
    9: "Billing & Subscription Errors",
    10: "Investment Analytics Optimization",
    11: "System Crashes & Compatibility",
    12: "SaaS Platform Performance",
    13: "Financial Data Discrepancies",
    14: "Server Outages & Peak Load"
}

de_labels = {
    0: "Software Bug Reports",
    1: "General/Miscellaneous (Noisy)",
    2: "VPN Connectivity Issues",
    3: "System Crashes (Data Platform)",
    4: "Digital Marketing Strategy",
    5: "Critical System Outages (Urgent)",
    6: "Billing & Payment Errors",
    7: "Medical Data Integrity",
    8: "Marketing Tool Performance",
    9: "SaaS Integration Support",
    10: "General Inquiry/Access Support",
    11: "Cybersecurity & Firewall",
    12: "Maintenance & Update Delays",
    13: "Investment Analytics Improvement",
    14: "Data Analysis Tool Bugs"
}

def apply_labels(lang, label_map):
    path = f'results/analysis/ranked_issues_{lang}.csv'
    df = pd.read_csv(path)
    df['business_label'] = df['cluster_id'].map(label_map)
    # Reorder columns to put label at front
    cols = ['cluster_id', 'business_label'] + [c for c in df.columns if c not in ['cluster_id', 'business_label']]
    df = df[cols]
    df.to_csv(path, index=False)
    print(f"Applied business labels to {lang} ranked issues.")

def issue_label(row):
    business_label = row.get('business_label')
    if pd.notna(business_label) and str(business_label).strip():
        return str(business_label)
    return str(row['top_words'])[:60] + "..."

def write_summary():
    en_ranked = pd.read_csv('results/analysis/ranked_issues_en.csv')
    de_ranked = pd.read_csv('results/analysis/ranked_issues_de.csv')

    with open('results/analysis/summary_comparison.txt', 'w', encoding='utf-8') as f:
        f.write("Task 4: Mining Insights Summary\n")
        f.write("===============================\n\n")
        f.write("Top 5 Issues in English (by Actionability Score):\n")
        for _, row in en_ranked.head(5).iterrows():
            f.write(
                f"- Cluster {row['cluster_id']}: {issue_label(row)} "
                f"(Score: {row['actionability_score']:.1f})\n"
            )

        f.write("\nTop 5 Issues in German (by Actionability Score):\n")
        for _, row in de_ranked.head(5).iterrows():
            f.write(
                f"- Cluster {row['cluster_id']}: {issue_label(row)} "
                f"(Score: {row['actionability_score']:.1f})\n"
            )


## Rank Issues

In [7]:
import pandas as pd
#from update_labels import apply_labels, en_labels, de_labels

priority_map = {
    'Low': 1,
    'Medium': 2,
    'High': 3,
    'Critical': 4
}

def issue_label(row):
    business_label = row.get('business_label')
    if pd.notna(business_label) and str(business_label).strip():
        return str(business_label)
    return str(row['top_words'])[:60] + "..."

def write_summary():
    en_ranked = pd.read_csv('results/analysis/ranked_issues_en.csv')
    de_ranked = pd.read_csv('results/analysis/ranked_issues_de.csv')

    with open('results/analysis/summary_comparison.txt', 'w', encoding='utf-8') as f:
        f.write("Task 4: Mining Insights Summary\n")
        f.write("===============================\n\n")
        f.write("Top 5 Issues in English (by Actionability Score):\n")
        for _, row in en_ranked.head(5).iterrows():
            f.write(
                f"- Cluster {row['cluster_id']}: {issue_label(row)} "
                f"(Score: {row['actionability_score']:.1f})\n"
            )

        f.write("\nTop 5 Issues in German (by Actionability Score):\n")
        for _, row in de_ranked.head(5).iterrows():
            f.write(
                f"- Cluster {row['cluster_id']}: {issue_label(row)} "
                f"(Score: {row['actionability_score']:.1f})\n"
            )

def analyze_impact(lang, method):
    print(f"Analyzing impact for {lang} {method}...")
    df = pd.read_csv(f'results/interpretation/df_{lang}_{method}_labeled.csv')
    report = pd.read_csv(f'results/interpretation/cluster_report_{lang}_{method}.csv')
    
    # Map priority
    df['priority_val'] = df['priority'].map(priority_map).fillna(1)
    
    # Calculate impact metrics per cluster
    stats = df.groupby('cluster').agg({
        'priority_val': 'mean',
        'subject': 'count' 
    }).rename(columns={'priority_val': 'avg_priority', 'subject': 'volume'})
    
    # Merge with cluster info
    final_report = report.merge(stats, left_on='cluster_id', right_index=True)
    
    # Calculate "Actionability Score" (Volume * Avg Priority)
    final_report['actionability_score'] = final_report['volume'] * final_report['avg_priority']
    
    # Rank by score
    final_report = final_report.sort_values(by='actionability_score', ascending=False)
    
    # Save final ranked report
    final_report.to_csv(f'results/analysis/ranked_issues_{lang}.csv', index=False)
    
    return final_report

print("Running impact analysis...")
analyze_impact('en', 'emb')
analyze_impact('de', 'emb')
apply_labels('en', en_labels)
apply_labels('de', de_labels)
write_summary()

print("Analysis and ranking complete.")


Running impact analysis...
Analyzing impact for en emb...
Analyzing impact for de emb...
Applied business labels to en ranked issues.
Applied business labels to de ranked issues.
Analysis and ranking complete.


In [8]:

if __name__ == "__main__":
    apply_labels('en', en_labels)
    apply_labels('de', de_labels)
    write_summary()


Applied business labels to en ranked issues.
Applied business labels to de ranked issues.
